# Article eligibility and user profiles

BT4222 Group 6 · representations 3/3. Run `3_01_news_embeddings.ipynb` and `3_02_entity_embeddings.ipynb` first.

Build observation-time article eligibility, supplied-history profiles for baselines/clustering, and demonstrate
causal profiles with long-term, recent and up to three category-interest components. Actual request-time
history references are exported by the feature notebook during chronological replay.

Run top to bottom. Shared algorithms, validation and file handling live in `src/newsrec/`; this notebook controls settings and shows results. You do not run those Python files separately. Saved artifacts and `REBUILD` work as before.

In [ ]:
from pathlib import Path
import os
import sys
import json
import numpy as np
import pandas as pd
from IPython.display import display

# Find the source package whether this notebook starts in the root or its subfolder.
CODE_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents,
                            Path('/content/drive/MyDrive/BT4222'))
                  if (p / 'src/newsrec').is_dir()), None)
if CODE_ROOT is None:
    raise FileNotFoundError('Open this notebook from the project checkout containing src/newsrec/.')
sys.path.insert(0, str(CODE_ROOT / 'src'))
from newsrec.common import Settings, saved_stage_ready

DATA_ROOT = None  # Optional: point to another folder holding the project's saved datasets.
ROOT = Path(DATA_ROOT or os.environ.get('NEWSREC_ROOT') or CODE_ROOT).expanduser().resolve()
ARTIFACTS = ROOT / '3_Representations/artifacts'
print('Data/output root:', ROOT)

from newsrec.representations import build_availability
from newsrec.pipeline import build_supplied_profiles, profile_preview

In [ ]:
# Defaults live in src/newsrec/common.py; override with Settings(...) if needed.
SETTINGS = Settings()
REBUILD = False  # Reuse saved outputs; True regenerates this notebook's results.

## 1. Observation-time eligibility

`available_from` is the earliest observable slate appearance or supplied-history discovery, not publication time.
Do not backdate new test users' histories to training start. A history-only article has no observed exposure age.
Eligibility uses `available_from <= t`; it is separate from counts built from events strictly before `t`.

In [ ]:
if REBUILD or not saved_stage_ready(ROOT, SETTINGS, "availability"):
    availability = build_availability(ROOT, SETTINGS)
else:
    print("Loaded saved article availability.")
    availability = pd.read_parquet(ARTIFACTS / "article_availability.parquet")
display(availability.head())
display(availability.discovery_source.value_counts())

## 2. Supplied-history profiles

Pool every valid supplied-history article vector (no time decay because click times are unknown).
These matrices never consume logged clicks. They are appropriate for the report's clustering experiment:
fit a scaler/K-means only on rows with `seen_in_train=True`; freeze both for later users.
A user absent from training can still have a usable supplied history.

In [ ]:
if REBUILD or not saved_stage_ready(ROOT, SETTINGS, "profiles"):
    profiles = build_supplied_profiles(ROOT, SETTINGS)
else:
    print("Loaded saved supplied-history profiles.")
    profiles = pd.read_parquet(ARTIFACTS / "supplied_profile_index.parquet")
display(
    profiles.groupby("seen_in_train").agg(
        users=("user_code", "size"),
        no_history=("history_length", lambda x: int(x.eq(0).sum())),
        no_entity_profile=("valid_entity_history_count", lambda x: int(x.eq(0).sum())),
    )
)
display(profiles.head())

## 3. Demonstrate causal profile updates

Seed each user once; append genuine earlier click events, retaining repeats. Score all requests in a timestamp
batch before consuming its feedback. Full category counts use all prior clicks; vector/NRMS histories are capped.
Snapshot chronology is unverified by default, so its capped subset is deterministic and not called recent.
Recent profiles use timestamped in-window clicks only. Up to three category interests are ranked by full
prior category counts and pooled from valid capped title vectors. Empty groups are masked.

The example processes only three initial timestamp batches, with their preceding state intact.

In [ ]:
display(profile_preview(ROOT, SETTINGS))
display(pd.read_csv(ROOT / 'results/representation_coverage.csv'))

## Handoff

The shared `TemporalState` class exposes causal text/entity/recent profiles, category counts, interest vectors,
and padded history-row sequences/masks. The feature build writes per-request history references so
retrieval, attention rankers and future ablations can reconstruct the same profile. State is timestamp-scoped;
a final user vector must never be reused for an earlier request. The disk pair store is replay scratch space,
not a resumable checkpoint. Persisted resume support is not implemented in this version.